# Instructions for this notebook:

Prior to running this notebook, the following data must be downloaded and installed in the project folder under Data/Raw/manning/:

[CLRH Hydrofabric dataset](https://hydrology.uwaterloo.ca/CLRH/Hydrofabric.html) (download watershed of interest based on outlet point) 
- Extract zip file to: /Data/Raw/manning/WB_Data/

[NHN streams dataset](https://ftp.maps.canada.ca/pub/nrcan_rncan/vector/geobase_nhn_rhn/shp_en/) (download shapefile corresponding to outlet point ID (e.g., 05OE000)) 
- Extract zip file to: /Data/Raw/manning/WB_Data/

[LiDAR DEM](https://mli.gov.mb.ca/dems/index_external_lidar.html) (download watershed of interest) ***CLIPPED AND RESAMPLED IN ARCGIS PRO
- Extract zip file to: /Data/Raw/manning/WB_Data/

# Calculate Depression Depths (mm) per Subbasin

In [1]:
import os
import sys
import fileinput
import numpy as np
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point, LineString, Polygon, MultiPoint
from rasterio.crs import CRS
import rasterio
from bs4 import BeautifulSoup

sys.path.append("..")
from src.py.proj_conf import *

## Import input files

In [2]:
os.getcwd()

'c:\\Users\\emilito\\Documents\\PWA\\_notebooks'

In [3]:
# CLRH basin delineation shapefile
clrh_gdf = gpd.read_file(DATA_RAW + "WB_Data/finalcat_info_v1-0.shp")

In [22]:
# CLRH basin delineation shapefile
#clrh_gdf = gpd.read_file(os.path.abspath(os.path.join(os.path.dirname("__file__"), '..', 'Data/Raw/manning/WB_Data')) + "/" + "finalcat_info_v1-0.shp")

In [4]:
# Project clrh basins and write to shapefile
clrh_gdf_projected_4617 = clrh_gdf.to_crs(4617)
clrh_gdf_projected_26914 = clrh_gdf.to_crs(26914)

clrh_gdf_projected_4617.to_file(DATA_RAW + "WB_Data/clrh_basins_projected_4617.shp")
clrh_gdf_projected_26914.to_file(DATA_RAW + "WB_Data/clrh_basins_projected_26914.shp")

c:\Users\emilito\AppData\Local\anaconda3\envs\pwa_dev\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Value 310011982 of field Seg_ID of feature 0 not successfully written. Possibly due to too larger number with respect to field width
  ogr_write(
c:\Users\emilito\AppData\Local\anaconda3\envs\pwa_dev\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Value 310002444 of field Seg_ID of feature 1 not successfully written. Possibly due to too larger number with respect to field width
  ogr_write(
c:\Users\emilito\AppData\Local\anaconda3\envs\pwa_dev\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Value 310002428 of field Seg_ID of feature 2 not successfully written. Possibly due to too larger number with respect to field width
  ogr_write(
c:\Users\emilito\AppData\Local\anaconda3\envs\pwa_dev\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Value 310011981 of field Seg_ID of feature 3 not successfully written. Possibly due to too larger number with respect to field wid

In [ ]:
# Project clrh basins and write to shapefile
#clrh_gdf_projected_4617 = clrh_gdf.to_crs(4617)
#clrh_gdf_projected_26914 = clrh_gdf.to_crs(26914)

#clrh_gdf_projected_4617.to_file(os.path.abspath(os.path.join(os.path.dirname("__file__"), '..', 'Data/Raw/manning/WB_Data')) + "/" + "clrh_basins_projected_4617.shp")
#clrh_gdf_projected_26914.to_file(os.path.abspath(os.path.join(os.path.dirname("__file__"), '..', 'Data/Raw/manning/WB_Data')) + "/" + "clrh_basins_projected_26914.shp")

c:\Users\emilito\AppData\Local\anaconda3\envs\pwa_dev\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Value 310011982 of field Seg_ID of feature 0 not successfully written. Possibly due to too larger number with respect to field width
  ogr_write(
c:\Users\emilito\AppData\Local\anaconda3\envs\pwa_dev\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Value 310002444 of field Seg_ID of feature 1 not successfully written. Possibly due to too larger number with respect to field width
  ogr_write(
c:\Users\emilito\AppData\Local\anaconda3\envs\pwa_dev\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Value 310002428 of field Seg_ID of feature 2 not successfully written. Possibly due to too larger number with respect to field width
  ogr_write(
c:\Users\emilito\AppData\Local\anaconda3\envs\pwa_dev\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Value 310011981 of field Seg_ID of feature 3 not successfully written. Possibly due to too larger number with respect to field wid

## Import WhiteboxTools

In [5]:
from WBT.whitebox_tools import WhiteboxTools

wbt = WhiteboxTools()

In [6]:
wbt.set_working_dir(os.path.abspath(os.path.join(os.path.dirname("__file__"),  '..', 'Data/Raw/manning/WB_Data')))

In [7]:
wbt.get_working_dir()

'c:\\Users\\emilito\\Documents\\PWA\\Data\\Raw\\manning\\WB_Data'

## Process DEM and Stream Network

In [8]:
### SUBBASIN AND STREAM SHAPEFILES WERE PROJECTED IN ARCGIS PRO (NO PROJECTION TOOL IN WHITEBOX)

# Clip NHN streams to study area
wbt.clip(
    i = 'NHN_05OE000_5_0_HD_SLWATER_1.shp',
    clip = 'clrh_basins_projected_4617.shp',
    output = 'NHN_Clip.shp'
)

### RASTER DATA WAS CLIPPED AND RESAMPLED IN ARCGIS PRO DUE TO MEMORY ALLOCATION ERROR

# Clip dem raster to study area
#wbt.clip_raster_to_polygon(
    #i = 'sr_dem_cgvd28.tif',
    #polygons = 'finalcat_info_v1-0_Project.shp',
    #output = 'sr_dem_cgvd28_Clip_MC.tif'
#)

# Resample DEM to 5m (for faster processing)
#wbt.resample(
    #inputs = 'sr_dem_cgvd28_Clip_MC.tif',
    #output = 'sr_dem_cgvd28_Clip_MC_Resample_5m_corr',
    #cell_size = 5,
    #method = "cc"
#)

.\whitebox_tools.exe --run="Clip" --wd="c:\Users\emilito\Documents\PWA\Data\Raw\manning\WB_Data" --input='NHN_05OE000_5_0_HD_SLWATER_1.shp' --clip='clrh_basins_projected_4617.shp' --output='NHN_Clip.shp' -v --compress_rasters=False

****************************
* Welcome to Clip          *
* Powered by WhiteboxTools *
* www.whiteboxgeo.com      *
****************************
Reading data...
Progress: 0%
Progress: 1%
Progress: 2%
Progress: 3%
Progress: 4%
Progress: 5%
Progress: 6%
Progress: 7%
Progress: 8%
Progress: 9%
Progress: 10%
Progress: 11%
Progress: 12%
Progress: 13%
Progress: 14%
Progress: 15%
Progress: 16%
Progress: 17%
Progress: 18%
Progress: 19%
Progress: 20%
Progress: 21%
Progress: 22%
Progress: 23%
Progress: 24%
Progress: 25%
Progress: 26%
Progress: 27%
Progress: 28%
Progress: 29%
Progress: 30%
Progress: 31%
Progress: 32%
Progress: 33%
Progress: 34%
Progress: 35%
Progress: 36%
Progress: 37%
Progress: 38%
Progress: 39%
Progress: 40%
Progress: 41%
Progress: 42%
Progress: 43%


0

In [9]:
os.getcwd()

### RUNNING WHITEBOX TOOLS CHANGES THE WORKING DIRECTORY (ADDS WBT FOLDER). NOTEBOOK MUST BE RUN TOP TO BOTTOM, OTHERWISE PATHS WITLL NOT WORK

'c:\\Users\\emilito\\Documents\\PWA\\_notebooks\\WBT'

In [10]:
# Project NHN clipped shapefile
streams_clip = gpd.read_file(os.path.abspath(os.path.join(os.path.dirname("__file__"), '..', '..', 'Data/Raw/manning/WB_Data')) + "/" + "NHN_Clip.shp")
streams_clip_proj_26914 = streams_clip.to_crs(26914)
streams_clip_proj_26914.to_file(os.path.abspath(os.path.join(os.path.dirname("__file__"), '..', '..', 'Data/Raw/manning/WB_Data')) + "/" + "NHN_Clip_Project_26914.shp")

## Generate Depressions Raster

In [11]:
# Burn streams into DEM and fill
wbt.fill_burn(
    dem = 'sr_dem_cgvd28_Clip_MC_Resample_5m_corr.tif',
    streams = 'NHN_Clip_Project_26914.shp',
    output = 'DEM_Clip_MC_Resample_5m_FillBurn.tif'
)

# Subtract raw DEM from filled DEM
wbt.raster_calculator(
    output = 'DEM_Clip_MC_Resample_5m_FillBurn_Deps.tif',
    statement = "'DEM_Clip_MC_Resample_5m_FillBurn.tif' - 'sr_dem_cgvd28_Clip_MC_Resample_5m_corr.tif'"
)

# Remove stray burn lines
wbt.conditional_evaluation(
    i = 'DEM_Clip_MC_Resample_5m_FillBurn_Deps.tif',
    output = 'DEM_Clip_MC_Resample_5m_FillBurn_Deps_Corr.tif',
    statement = "value < 0.0",
    true = 0.0,
    false = 'DEM_Clip_MC_Resample_5m_FillBurn_Deps.tif'
)

.\whitebox_tools.exe --run="FillBurn" --wd="c:\Users\emilito\Documents\PWA\Data\Raw\manning\WB_Data" --dem='sr_dem_cgvd28_Clip_MC_Resample_5m_corr.tif' --streams='NHN_Clip_Project_26914.shp' --output='DEM_Clip_MC_Resample_5m_FillBurn.tif' -v --compress_rasters=False

****************************
* Welcome to FillBurn      *
* Powered by WhiteboxTools *
* www.whiteboxgeo.com      *
****************************
Reading streams data...
Reading DEM data...
Rasterizing Streams: 0%
Rasterizing Streams: 1%
Rasterizing Streams: 2%
Rasterizing Streams: 3%
Rasterizing Streams: 4%
Rasterizing Streams: 5%
Rasterizing Streams: 6%
Rasterizing Streams: 7%
Rasterizing Streams: 8%
Rasterizing Streams: 9%
Rasterizing Streams: 10%
Rasterizing Streams: 11%
Rasterizing Streams: 12%
Rasterizing Streams: 13%
Rasterizing Streams: 14%
Rasterizing Streams: 15%
Rasterizing Streams: 16%
Rasterizing Streams: 17%
Rasterizing Streams: 18%
Rasterizing Streams: 19%
Rasterizing Streams: 20%
Rasterizing Streams: 21%
Ras

0

## Calculate Depression Depths per Subbasin

In [12]:
# Convert clrh subbasin polygons to raster
wbt.vector_polygons_to_raster(
    i = 'clrh_basins_projected_26914.shp',
    output = 'clrh_basins_raster.tif',
    field = "FID",
    nodata = True,
    cell_size = 5.0
)

# Define projection of clrh raster
with rasterio.open(os.path.abspath(os.path.join(os.path.dirname("__file__"), '..', '..', 'Data/Raw/manning/WB_Data')) + "/" + 'clrh_basins_raster.tif', 'r+') as rds:
    rds.crs = CRS.from_epsg(26914)

# Calculate zonal statistics
wbt.zonal_statistics(
    i = 'DEM_Clip_MC_Resample_5m_FillBurn_Deps_Corr.tif',
    features = 'clrh_basins_raster.tif',
    stat="total",
    out_table='ZonalStats_DEM_Clip_MC_Resample_5m_FillBurn_Deps_Corr.html'
)

.\whitebox_tools.exe --run="VectorPolygonsToRaster" --wd="c:\Users\emilito\Documents\PWA\Data\Raw\manning\WB_Data" --input='clrh_basins_projected_26914.shp' --field=FID --output='clrh_basins_raster.tif' --nodata --cell_size='5.0' -v --compress_rasters=False

*************************************
* Welcome to VectorPolygonsToRaster *
* Powered by WhiteboxTools          *
* www.whiteboxgeo.com               *
*************************************
Reading data...
Reading attributes: 0%
Reading attributes: 1%
Reading attributes: 3%
Reading attributes: 5%
Reading attributes: 6%
Reading attributes: 8%
Reading attributes: 10%
Reading attributes: 11%
Reading attributes: 13%
Reading attributes: 15%
Reading attributes: 16%
Reading attributes: 18%
Reading attributes: 20%
Reading attributes: 21%
Reading attributes: 23%
Reading attributes: 25%
Reading attributes: 26%
Reading attributes: 28%
Reading attributes: 30%
Reading attributes: 31%
Reading attributes: 33%
Reading attributes: 35%
Reading attri

0

In [13]:
# Read html file as pandas dataframe
file = os.path.abspath(os.path.join(os.path.dirname("__file__"), '..', '..', 'Data/Raw/manning/WB_Data')) + "/" + 'ZonalStats_DEM_Clip_MC_Resample_5m_FillBurn_Deps_Corr.html'
zonal_stats_dfs = pd.read_html(file, flavor='bs4')
zonal_stats_df = zonal_stats_dfs[0]

# Join totals to clrh subbasins geodataframe
clrh_gdf_projected_26914['Deps_SUM'] = zonal_stats_df['Total']

# Calculate depression volumes (m3) and add field to gdf
clrh_gdf_projected_26914['Deps_Vol_m3'] = clrh_gdf_projected_26914['Deps_SUM'] * 5 * 5

# Calculate depression depths (mm) and add field to gdf
clrh_gdf_projected_26914['Deps_Depth_mm'] = clrh_gdf_projected_26914['Deps_Vol_m3'] / clrh_gdf_projected_26914['BasArea'] * 1000

In [14]:
# Display subbasins geodataframe with depression depths field
clrh_gdf_projected_26914

,SubId,DowSubId,RivSlope,RivLength,BasSlope,BasAspect,BasArea,BkfWidth,BkfDepth,Lake_Cat,...,outletLat,outletLng,k,c,SSDA_NM,SDA_NM,geometry,Deps_SUM,Deps_Vol_m3,Deps_Depth_mm
0,310013151,310015352,0.001359,1472.132034,0.139288,101.631184,1020600.0,3.598586,0.178092,0.0,...,49.408374,-96.609506,0.004507,0.985797,Nelson,05OS,"POLYGON ((673982.775 5475039.332, 673863.049 5...",461.1001,11527.5025,11.294829
1,310013152,310015352,0.000777,3362.132034,0.341033,113.693816,5556600.0,1.557162,0.107737,0.0,...,49.408101,-96.609915,0.004507,0.985797,Nelson,05OS,"POLYGON ((671497.946 5474216.278, 671527.877 5...",1696.0727,42401.8175,7.630893
2,310013155,310013151,0.000768,1301.543289,0.094106,59.975058,3028500.0,1.322273,0.100000,0.0,...,49.404947,-96.595011,0.004507,0.985797,Nelson,05OS,"MULTIPOLYGON (((676201.988 5477234.473, 676231...",1715.9481,42898.7025,14.165000
3,310013156,310013151,0.000496,6743.452378,0.136037,93.606178,18478800.0,3.324347,0.169820,0.0,...,49.404677,-96.595007,0.004507,0.985797,Nelson,05OS,"POLYGON ((678199.915 5471602.934, 678197.475 5...",8671.4900,216787.2500,11.731674
4,310013179,310013156,0.000032,796.690476,0.085525,40.444107,742500.0,1.155956,0.100000,0.0,...,49.379474,-96.561250,0.004507,0.985797,Nelson,05OS,"POLYGON ((676659.448 5471149.321, 676629.519 5...",238.2421,5956.0525,8.021620
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56,310020587,310013155,-1.234500,-1.234500,0.068989,45.713966,4674600.0,1.033715,0.100000,0.0,...,49.410416,-96.583526,0.004507,0.985797,Nelson,05OS,"POLYGON ((676497.646 5475863.467, 676496.425 5...",2779.3009,69482.5225,14.863843
57,310020603,310013152,-1.234500,-1.234500,0.270868,122.177060,5176800.0,1.087038,0.100000,0.0,...,49.388680,-96.605107,0.004507,0.985797,Nelson,05OS,"POLYGON ((674738.472 5471943.066, 674728.694 5...",2157.4525,53936.3125,10.418852
58,310020623,310013179,-1.234500,-1.234500,0.086642,64.575319,5121900.0,1.081340,0.100000,0.0,...,49.376452,-96.570287,0.004507,0.985797,Nelson,05OS,"POLYGON ((677000.78 5467915.547, 676821.217 54...",1584.6294,39615.7350,7.734578
59,310020644,310013184,-1.234500,-1.234500,0.122003,89.093174,8779500.0,1.410327,0.101521,0.0,...,49.360158,-96.535850,0.004507,0.985797,Nelson,05OS,"POLYGON ((679774.133 5466795.359, 679772.915 5...",3303.8899,82597.2475,9.407967
